# Calculate invoice


In [1]:

import csv
import json
import re
from collections import defaultdict
from datetime import datetime
from decimal import Decimal, ROUND_CEILING
from pathlib import Path
from zoneinfo import ZoneInfo

UK = ZoneInfo("Europe/London")

In [2]:

def parse_decimal_hours_to_minutes(value):
    value = value.strip()
    if ":" in value:
        h, m = value.split(":", 1)
        return Decimal(int(h) * 60 + int(m))
    return Decimal(value) * 60

def ceil_to_quarter_hour(minutes):
    if not isinstance(minutes, Decimal):
        minutes = Decimal(str(minutes))
    quarters = (minutes / Decimal(15)).to_integral_value(rounding=ROUND_CEILING)
    return int(quarters * 15)

def duration_to_minutes(duration):
    match = re.fullmatch(
        r"P(?:(?P<days>\d+)D)?T(?:(?P<hours>\d+)H)?(?:(?P<minutes>\d+)M)?(?:(?P<seconds>\d+)S)?",
        duration.strip(),
    )
    if not match:
        raise ValueError(f"Unsupported ISO duration: {duration!r}")
    parts = match.groupdict()
    total = Decimal(0)
    total += Decimal(parts["days"] or 0) * 24 * 60
    total += Decimal(parts["hours"] or 0) * 60
    total += Decimal(parts["minutes"] or 0)
    total += Decimal(parts["seconds"] or 0) / 60
    return total

def load_staff(path):
    staff_by_email = {}
    staff_by_initials = {}
    with path.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            staff_by_email[row["email"].strip().lower()] = row["name"].strip()
            staff_by_initials[row["initials"].strip()] = row["name"].strip()
    def resolve(identifier):
        identifier = identifier.strip()
        if identifier.lower() in staff_by_email:
            return staff_by_email[identifier.lower()]
        if identifier in staff_by_initials:
            return staff_by_initials[identifier]
        return identifier
    return resolve

def load_rates(path):
    rates = defaultdict(list)
    with path.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            rates[row["consultant"].strip()].append(
                (row["effective_from"].strip(), Decimal(row["hourly_rate_gbp"].strip()))
            )
    for consultant in rates:
        rates[consultant].sort()
    def rate_for(consultant, work_date):
        applicable = None
        for effective_from, rate in rates[consultant.strip()]:
            if effective_from <= work_date.isoformat():
                applicable = (effective_from, rate)
        if applicable is None:
            raise ValueError(f"No applicable rate for {consultant} on {work_date}")
        return applicable[1]
    return rate_for

def load_projects(path):
    target_projects = set()
    with path.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            code = row["code"].strip().upper()
            if row["client"].strip() == "Northwind Partners" and row["billing"].strip() == "time":
                target_projects.add(code)
    def is_target(project_code):
        return project_code.strip().upper() in target_projects
    return is_target

def load_ledgerone(path, rate_for, is_target):
    entries = []
    seen_ids = set()
    with path.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            entry_id = row["entry_id"].strip()
            if entry_id in seen_ids:
                continue
            seen_ids.add(entry_id)
            work_date = datetime.strptime(row["date"].strip(), "%d/%m/%Y").date()
            if not (work_date.year == 2026 and work_date.month == 9):
                continue
            project = row["project"].strip().upper()
            if not is_target(project):
                continue
            consultant = row["consultant"].strip()
            if row["billable"].strip().lower() not in {"y", "yes", "true", "1"}:
                continue
            minutes = parse_decimal_hours_to_minutes(row["hours"])
            billed_minutes = ceil_to_quarter_hour(minutes)
            rate = rate_for(consultant, work_date)
            pence = (Decimal(billed_minutes) / 60) * rate * 100
            entries.append(pence)
    return entries

def load_clockwise(path, resolve_staff, rate_for, is_target):
    data = json.loads(path.read_text(encoding="utf-8"))
    by_id = defaultdict(list)
    for entry in data["entries"]:
        by_id[entry["id"]].append(entry)
    chosen = [max(versions, key=lambda e: e.get("version", 1)) for versions in by_id.values()]
    entries = []
    for entry in chosen:
        if entry.get("deleted"):
            continue
        start = datetime.fromisoformat(entry["start"].replace("Z", "+00:00"))
        work_date = start.astimezone(UK).date()
        if not (work_date.year == 2026 and work_date.month == 9):
            continue
        project = entry["project"].strip().upper()
        if not is_target(project):
            continue
        if not bool(entry.get("billable")):
            continue
        consultant = resolve_staff(entry["user"])
        billed_minutes = ceil_to_quarter_hour(duration_to_minutes(entry["duration"]))
        rate = rate_for(consultant, work_date)
        pence = (Decimal(billed_minutes) / 60) * rate * 100
        entries.append(pence)
    return entries

def load_timebox(path, resolve_staff, rate_for, is_target):
    entries = []
    with path.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            imported_from = row["Imported From"].strip()
            if imported_from:
                continue
            work_date = datetime.strptime(row["Date"].strip(), "%m/%d/%Y").date()
            if not (work_date.year == 2026 and work_date.month == 9):
                continue
            project = row["Project"].strip().upper()
            if not is_target(project):
                continue
            consultant = resolve_staff(row["Staff"])
            billed_minutes = ceil_to_quarter_hour(Decimal(row["Minutes"].strip()))
            rate = rate_for(consultant, work_date)
            pence = (Decimal(billed_minutes) / 60) * rate * 100
            entries.append(pence)
    return entries

In [3]:
data_dir = Path(".")
resolve_staff = load_staff(Path("/home/shay/Downloads/leaky-ledger/staff.csv"))
rate_for = load_rates(Path("/home/shay/Downloads/leaky-ledger/rates.csv"))
is_target = load_projects(Path("/home/shay/Downloads/leaky-ledger/projects.csv"))
ledgerone_pence = load_ledgerone(Path("/home/shay/Downloads/leaky-ledger/ledgerone.csv"), rate_for, is_target)
clockwise_pence = load_clockwise(Path("/home/shay/Downloads/leaky-ledger/clockwise.json"), resolve_staff, rate_for, is_target)
timebox_pence = load_timebox(Path("/home/shay/Downloads/leaky-ledger/timebox.csv"), resolve_staff, rate_for, is_target)
all_pence = ledgerone_pence + clockwise_pence + timebox_pence
total = sum(all_pence, Decimal(0))
print(f"Ledgerone entries: {len(ledgerone_pence)}")
print(f"Clockwise entries: {len(clockwise_pence)}")
print(f"Timebox entries:   {len(timebox_pence)}")
print(f"Total pence:      {int(total)}")

Ledgerone entries: 43
Clockwise entries: 112
Timebox entries:   46
Total pence:      7444875
